# Librerías

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)
import os

os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

# Sesión de Spark

In [3]:
spark = (
    SparkSession.builder
    .appName("Lab7_ENEIC")
    .getOrCreate()
)
print("Versión de Spark:", spark.version)

Versión de Spark: 3.5.8


# Setup

In [8]:
archivos = {
    "2025T1": "datos/Personas_ENEIC_T1_2025.xlsx",
    "2025T2": "datos/Personas-ENEIC-T2-2025.xlsx",
    "2025T3": "datos/Base-de-datos-Personas-ENEIC-III-2025.xlsx",
    "2025T4": "datos/Base-de-datos-Personas-ENEIC-IV-2025.xlsx",
    "2026T1": "datos/Base-de-datos-Personas-ENEIC-I-2026.xlsx"
}

ruta_diccionario = "datos/Diccionario-Personas-ENEIC-I-2026.xlsx"

columnas = [
    "P05D01",       # salario mensual
    "P02A03",       # edad
    "P05C07A",      # antigüedad - años
    "P05C07B",      # antigüedad - meses
    "P05H01A",      # horas semanales
    "P03A03A",      # nivel educativo
    "P05C16",       # categoría ocupacional
    "DOMINIO",
    "OCUPADOS",
    "NUM_HOGAR",
    "NUM_PERSONA",
    "FACTOR",
    "ANIO",
    "TRIMESTRE"
]

ruta_t1_2025 = archivos["2025T1"]

excel_t1 = pd.ExcelFile(ruta_t1_2025)

print(excel_t1.sheet_names)

t1_2025_pd = pd.read_excel(
    ruta_t1_2025
)
print("Dimensiones:", t1_2025_pd.shape)

['Personas_ENEIC_T1_2025']
Dimensiones: (51588, 270)


In [9]:
t1_2025_pd[columnas].dtypes

P05D01         float64
P02A03           int64
P05C07A        float64
P05C07B        float64
P05H01A        float64
P03A03A        float64
P05C16         float64
DOMINIO          int64
OCUPADOS       float64
NUM_HOGAR        int64
NUM_PERSONA      int64
FACTOR           int64
ANIO             int64
TRIMESTRE        int64
dtype: object

In [12]:
t1_2025 = t1_2025_pd[columnas].copy()
renombres = {
    "P05D01": "salario_mensual",
    "P02A03": "edad",
    "P05C07A": "antiguedad_anios",
    "P05C07B": "antiguedad_meses",
    "P05H01A": "horas_semanales",
    "P03A03A": "nivel_educativo",
    "P05C16": "categoria_ocupacional",
    "DOMINIO": "dominio",
    "OCUPADOS": "ocupado"
}

t1_2025 = t1_2025.rename(columns=renombres)

t1_2025["archivo_origen"] = "Personas_ENEIC_T1_2025.xlsx"
t1_2025["periodo_archivo"] = "2025T1"
t1_2025["anio_archivo"] = 2025
t1_2025["trimestre_calendario"] = 1

t1_2025["antiguedad"] = (
    t1_2025["antiguedad_anios"]
    + t1_2025["antiguedad_meses"] / 12
)

t1_2025.head()

,salario_mensual,edad,antiguedad_anios,antiguedad_meses,horas_semanales,nivel_educativo,categoria_ocupacional,dominio,ocupado,NUM_HOGAR,NUM_PERSONA,FACTOR,ANIO,TRIMESTRE,archivo_origen,periodo_archivo,anio_archivo,trimestre_calendario,antiguedad
0,NaN,14,NaN,NaN,NaN,3.0,NaN,2,NaN,13796,3,338,2025,2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1,NaN
1,NaN,42,23.0,0.0,35.0,5.0,6.0,2,1.0,13796,1,338,2025,2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1,23.0
2,7000.0,42,20.0,0.0,1.0,5.0,1.0,2,1.0,13796,2,338,2025,2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1,20.0
3,NaN,9,NaN,NaN,NaN,2.0,NaN,2,NaN,13796,5,338,2025,2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1,NaN
4,NaN,11,NaN,NaN,NaN,2.0,NaN,2,NaN,13796,4,338,2025,2,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1,NaN


In [13]:
def cargar_periodo(ruta, periodo, anio, trimestre):

    # Leer Excel
    df = pd.read_excel(ruta)

    # Guardar cantidad original
    n_original = len(df)

    # Seleccionar únicamente las columnas necesarias
    df = df[columnas].copy()

    # Renombrar variables analíticas
    df = df.rename(columns=renombres)

    # Agregar trazabilidad del archivo
    df["archivo_origen"] = ruta
    df["periodo_archivo"] = periodo
    df["anio_archivo"] = anio
    df["trimestre_calendario"] = trimestre

    # Homologar variables numéricas
    columnas_numericas = [
        "salario_mensual",
        "edad",
        "antiguedad_anios",
        "antiguedad_meses",
        "horas_semanales",
        "FACTOR"
    ]

    for col in columnas_numericas:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # Homologar códigos categóricos como strings
    columnas_categoricas = [
        "nivel_educativo",
        "categoria_ocupacional",
        "dominio",
        "ocupado"
    ]

    for col in columnas_categoricas:
        df[col] = (
            pd.to_numeric(df[col], errors="coerce")
            .astype("Int64")
            .astype("string")
        )

    # IDs para auditoría
    df["NUM_HOGAR"] = (
        df["NUM_HOGAR"]
        .astype("Int64")
        .astype("string")
    )

    df["NUM_PERSONA"] = (
        df["NUM_PERSONA"]
        .astype("Int64")
        .astype("string")
    )

    # Construir antigüedad en años
    df["antiguedad"] = (
        df["antiguedad_anios"]
        + df["antiguedad_meses"] / 12
    )

    return df, n_original

In [14]:
t1_2025, n_t1_2025 = cargar_periodo(
    archivos["2025T1"],
    "2025T1",
    2025,
    1
)

t2_2025, n_t2_2025 = cargar_periodo(
    archivos["2025T2"],
    "2025T2",
    2025,
    2
)

t3_2025, n_t3_2025 = cargar_periodo(
    archivos["2025T3"],
    "2025T3",
    2025,
    3
)

t4_2025, n_t4_2025 = cargar_periodo(
    archivos["2025T4"],
    "2025T4",
    2025,
    4
)

t1_2026, n_t1_2026 = cargar_periodo(
    archivos["2026T1"],
    "2026T1",
    2026,
    1
)

In [26]:
schema_eneic = StructType([
    StructField("salario_mensual", DoubleType(), True),
    StructField("edad", DoubleType(), True),
    StructField("antiguedad_anios", DoubleType(), True),
    StructField("antiguedad_meses", DoubleType(), True),
    StructField("horas_semanales", DoubleType(), True),

    StructField("nivel_educativo", StringType(), True),
    StructField("categoria_ocupacional", StringType(), True),
    StructField("dominio", StringType(), True),
    StructField("ocupado", StringType(), True),

    StructField("NUM_HOGAR", StringType(), True),
    StructField("NUM_PERSONA", StringType(), True),

    StructField("FACTOR", DoubleType(), True),

    StructField("ANIO", IntegerType(), True),
    StructField("TRIMESTRE", IntegerType(), True),

    StructField("archivo_origen", StringType(), True),
    StructField("periodo_archivo", StringType(), True),

    StructField("anio_archivo", IntegerType(), True),
    StructField("trimestre_calendario", IntegerType(), True),

    StructField("antiguedad", DoubleType(), True)
])

In [27]:
def preparar_para_spark(df):

    df = df.copy()

    # Variables continuas
    columnas_double = [
        "salario_mensual",
        "edad",
        "antiguedad_anios",
        "antiguedad_meses",
        "horas_semanales",
        "FACTOR",
        "antiguedad"
    ]

    for col in columnas_double:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        df[col] = df[col].apply(
            lambda x: float(x) if pd.notna(x) else None
        )

    # Variables enteras
    columnas_int = [
        "ANIO",
        "TRIMESTRE",
        "anio_archivo",
        "trimestre_calendario"
    ]

    for col in columnas_int:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        df[col] = df[col].apply(
            lambda x: int(x) if pd.notna(x) else None
        )

    # Variables de texto
    columnas_string = [
        "nivel_educativo",
        "categoria_ocupacional",
        "dominio",
        "ocupado",
        "NUM_HOGAR",
        "NUM_PERSONA",
        "archivo_origen",
        "periodo_archivo"
    ]

    for col in columnas_string:
        df[col] = df[col].apply(
            lambda x: str(x) if pd.notna(x) else None
        )

    return df

In [28]:
t1_2025_spark_ready = preparar_para_spark(t1_2025)

fila = t1_2025_spark_ready.iloc[0]

for columna in t1_2025_spark_ready.columns:
    print(
        columna,
        repr(fila[columna]),
        type(fila[columna])
    )

salario_mensual np.float64(nan) <class 'numpy.float64'>
edad np.float64(14.0) <class 'numpy.float64'>
antiguedad_anios np.float64(nan) <class 'numpy.float64'>
antiguedad_meses np.float64(nan) <class 'numpy.float64'>
horas_semanales np.float64(nan) <class 'numpy.float64'>
nivel_educativo '3' <class 'str'>
categoria_ocupacional None <class 'NoneType'>
dominio '2' <class 'str'>
ocupado None <class 'NoneType'>
NUM_HOGAR '13796' <class 'str'>
NUM_PERSONA '3' <class 'str'>
FACTOR np.float64(338.0) <class 'numpy.float64'>
ANIO np.int64(2025) <class 'numpy.int64'>
TRIMESTRE np.int64(2) <class 'numpy.int64'>
archivo_origen 'datos/Personas_ENEIC_T1_2025.xlsx' <class 'str'>
periodo_archivo '2025T1' <class 'str'>
anio_archivo np.int64(2025) <class 'numpy.int64'>
trimestre_calendario np.int64(1) <class 'numpy.int64'>
antiguedad np.float64(nan) <class 'numpy.float64'>


In [31]:
spark_t1_2025 = spark.createDataFrame(
    t1_2025_spark_ready.to_dict("records"),
    schema=schema_eneic
)

spark_t2_2025 = spark.createDataFrame(
    preparar_para_spark(t2_2025).to_dict("records"),
    schema=schema_eneic
)

spark_t3_2025 = spark.createDataFrame(
    preparar_para_spark(t3_2025).to_dict("records"),
    schema=schema_eneic
)

spark_t4_2025 = spark.createDataFrame(
    preparar_para_spark(t4_2025).to_dict("records"),
    schema=schema_eneic
)

spark_t1_2026 = spark.createDataFrame(
    preparar_para_spark(t1_2026).to_dict("records"),
    schema=schema_eneic
)

datos_2025 = (
    spark_t1_2025
    .unionByName(spark_t2_2025)
    .unionByName(spark_t3_2025)
    .unionByName(spark_t4_2025)
)

In [32]:
variables_analisis = [
    "salario_mensual",
    "edad",
    "antiguedad_anios",
    "antiguedad_meses",
    "antiguedad",
    "horas_semanales",
    "nivel_educativo",
    "categoria_ocupacional",
    "dominio",
    "ocupado",
    "NUM_HOGAR",
    "NUM_PERSONA",
    "FACTOR"
]

total_2025 = datos_2025.count()

faltantes = []

for col in variables_analisis:
    n_faltantes = datos_2025.filter(
        F.col(col).isNull() |
        F.isnan(F.col(col)) if dict(datos_2025.dtypes)[col] in ["double", "float"]
        else F.col(col).isNull()
    ).count()

    faltantes.append(
        (
            col,
            n_faltantes,
            round(n_faltantes / total_2025 * 100, 2)
        )
    )

faltantes_df = spark.createDataFrame(
    faltantes,
    ["variable", "faltantes", "porcentaje"]
)

faltantes_df.show(truncate=False)

Py4JJavaError: An error occurred while calling o213.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 11 in stage 1.0 failed 1 times, most recent failure: Lost task 11.0 in stage 1.0 (TID 26) (localhost executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:203)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:109)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:124)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:174)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:67)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:367)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:331)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.s

In [ ]:
duplicados_clave = (
    datos_2025
    .groupBy(
        "periodo_archivo",
        "NUM_HOGAR",
        "NUM_PERSONA"
    )
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Claves duplicadas:",
    duplicados_clave.count()
)

duplicados_clave.show(10, truncate=False)

In [ ]:
resumen_filtros = []

df_filtrado = datos_2025

def aplicar_filtro(df, condicion, nombre):
    antes = df.count()

    nuevo = df.filter(condicion)

    despues = nuevo.count()

    resumen_filtros.append({
        "filtro": nombre,
        "antes": antes,
        "despues": despues,
        "excluidos": antes - despues
    })

    return nuevo

# 1. Edad válida
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("edad").isNotNull() &
    (~F.isnan("edad")) &
    (F.col("edad") >= 15),
    "Edad >= 15 y válida"
)

# 2. Ocupados
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("ocupado") == "1",
    "OCUPADOS = 1"
)

# 3. Asalariados
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("categoria_ocupacional").isin(
        ["1", "2", "3", "4"]
    ),
    "Categoría ocupacional 1-4"
)

# 4. Salario válido y positivo
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("salario_mensual").isNotNull() &
    (~F.isnan("salario_mensual")) &
    (F.col("salario_mensual") > 0),
    "Salario mensual > 0"
)

# 5. Años de antigüedad válidos
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("antiguedad_anios").isNotNull() &
    (~F.isnan("antiguedad_anios")) &
    (F.col("antiguedad_anios") >= 0),
    "Antigüedad en años válida"
)

# 6. Meses válidos
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("antiguedad_meses").isNotNull() &
    (~F.isnan("antiguedad_meses")) &
    (F.col("antiguedad_meses") >= 0) &
    (F.col("antiguedad_meses") <= 11) &
    (F.col("antiguedad_meses") == F.floor("antiguedad_meses")),
    "Meses de antigüedad entre 0 y 11"
)

# 7. Antigüedad <= edad
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("antiguedad").isNotNull() &
    (~F.isnan("antiguedad")) &
    (F.col("antiguedad") >= 0) &
    (F.col("antiguedad") <= F.col("edad")),
    "Antigüedad <= edad"
)

# 8. Horas válidas
df_filtrado = aplicar_filtro(
    df_filtrado,
    F.col("horas_semanales").isNotNull() &
    (~F.isnan("horas_semanales")) &
    (F.col("horas_semanales") > 0) &
    (F.col("horas_semanales") <= 168),
    "0 < horas semanales <= 168"
)

resumen_filtros_pd = pd.DataFrame(
    resumen_filtros
)

resumen_filtros_pd

print(
    "Registros originales 2025:",
    total_2025
)

print(
    "Registros analíticos finales:",
    df_filtrado.count()
)

datos_2025_filtrados = df_filtrado

datos_2025_filtrados.groupBy(
    "periodo_archivo"
).count().orderBy(
    "periodo_archivo"
).show()